[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/composite_scoring/composite_scoring.ipynb)

# Combine atomic scores with your own weights

"Is this applicant worth a call?" mixes the stack, the writing and the tone. One score would bake the mixture into the question, and a new priority would mean a new prompt. FRIDA-Decisions scores each factor on its own scale. The weights, and the cutoff that means "invite", are ordinary numbers.

Pattern: [Composite scoring](https://docs.typesafe.ai/patterns/composite-scoring). The letters are synthetic and short. The cutoff check shows that the number does something sensible. It is not a hiring metric.


In [1]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


Three scores, two weight vectors, one cutoff. `score` is an expected level from 0 to 4. On this rubric the letters land in the lower half, so the invite cutoff is 1.15, not the midpoint.


In [3]:
DATA = [
    {"name": "Ира", "invite": True, "text": "Три года писала бэкенд на Django и PostgreSQL. В прошлом квартале сама разобрала инцидент с блокировками в базе: время ответа вернулось с 800 мс к 90 мс. Могу показать разбор."},
    {"name": "Олег", "invite": True, "text": "Пять лет на Python. Django, PostgreSQL, очереди. Последний инцидент: реплика отставала на час, нашёл тяжёлый запрос и вынес его из транзакции. Пишу коротко, детали готов обсудить."},
    {"name": "Маша", "invite": False, "text": "Я повар. Ищу работу на кухне ресторана. Python, Django и базы данных мне не нужны, я ими не занималась."},
    {"name": "Кирилл", "invite": False, "text": "Делал вёрстку на React. С Python знаком по учебному курсу, базу данных в проде не сопровождал. Хочу перейти в бэкенд."},
    {"name": "Нина", "invite": False, "text": "Django видела, PostgreSQL ставила локально. Инциденты не разбирала, в проде не дежурила. Письмо длинное, потому что хочу рассказать про хобби и путешествия, это важно для культуры."},
    {"name": "Павел", "invite": True, "text": "Четыре года поддерживал Django-монолит и PostgreSQL на 2 ТБ. Ночные инциденты: дважды откатывал миграцию, один раз чинил дедлок по логам. Без воды: что сломалось, что сделал, чем кончилось."},
]
ROLE = "Вакансия: бэкенд на Python, нужны Django, PostgreSQL и опыт разбора чужих инцидентов."
RUBRICS = {
    "skills": [
        "Опыта по делу нет.",
        "Смежный опыт, нужного стека почти нет.",
        "Часть стека есть, без конкретных задач.",
        "Нужный стек и названы конкретные задачи.",
        "Стек, цифры и самостоятельные решения.",
    ],
    "clarity": [
        "Непонятно, что человек умеет.",
        "Общие слова, почти без фактов.",
        "Факты есть, но их приходится вычитывать.",
        "Письмо по делу и его легко дочитать.",
        "Коротко, по делу, с примерами.",
    ],
    "tone": [
        "Грубо или пренебрежительно.",
        "Фамильярно, рабочему письму не подходит.",
        "Нейтрально, без явных сбоев.",
        "Спокойно и уважительно.",
        "Уважительно и без заискивания.",
    ],
}
WEIGHTS = {
    "stack": {"skills": 0.7, "clarity": 0.2, "tone": 0.1},
    "writing": {"skills": 0.2, "clarity": 0.5, "tone": 0.3},
}
CUTOFF = 1.15


def request_for(text):
    return {
        "state": ROLE + "\n\nПисьмо кандидата:\n" + text,
        "questions": {
            "skills": {"type": "score", "instructions": "Насколько опыт совпадает с вакансией?",
                       "criteria": RUBRICS["skills"]},
            "clarity": {"type": "score", "instructions": "Насколько письмо ясное и конкретное?",
                        "criteria": RUBRICS["clarity"]},
            "tone": {"type": "score", "instructions": "Насколько тон уместен для рабочего письма?",
                     "criteria": RUBRICS["tone"]},
        },
    }


def weighted(answers, weights):
    total = 0.0
    for name in weights:
        total += weights[name] * answers[name]["score"]
    return total


Both columns use the same three answers. Only the weights change, and with them who clears the cutoff.


In [4]:
def run(judge):
    scored = []
    for row in DATA:
        answers = judge(request_for(row["text"]))["answers"]
        values = {}
        for name, weights in WEIGHTS.items():
            values[name] = weighted(answers, weights)
        scored.append({"name": row["name"], "invite": row["invite"], "answers": answers, "values": values})

    print(f"{'name':10} {'skills':6} {'clarity':7} {'tone':6} {'stack':7} {'writing':7} invite")
    hits = 0
    for row in scored:
        answers = row["answers"]
        invite = row["values"]["stack"] >= CUTOFF
        if invite == row["invite"]:
            hits += 1
        print(f"{row['name']:10} {answers['skills']['score']:.2f}   {answers['clarity']['score']:.2f}    "
              f"{answers['tone']['score']:.2f}   {row['values']['stack']:.2f}    "
              f"{row['values']['writing']:.2f}    {str(invite):5} gold {row['invite']}")

    by_stack = sorted(scored, key=lambda row: -row["values"]["stack"])
    by_writing = sorted(scored, key=lambda row: -row["values"]["writing"])
    print("\norder by stack weights:   " + ", ".join(row["name"] for row in by_stack))
    print("order by writing weights: " + ", ".join(row["name"] for row in by_writing))
    print(f"cutoff {CUTOFF} on the stack weights agrees with the invite label in {hits}/{len(DATA)}")


In [5]:
run(judge)


name       skills clarity tone   stack   writing invite
Ира        1.41   0.94    3.05   1.48    1.67    True  gold True
Олег       1.37   1.54    3.07   1.58    1.97    True  gold True
Маша       0.80   0.13    1.02   0.69    0.53    False gold False
Кирилл     0.85   0.24    1.52   0.79    0.75    False gold False
Нина       0.72   0.27    1.23   0.68    0.65    False gold False
Павел      1.32   0.72    2.19   1.29    1.28    True  gold True

order by stack weights:   Олег, Ира, Павел, Кирилл, Маша, Нина
order by writing weights: Олег, Ира, Павел, Кирилл, Нина, Маша
cutoff 1.15 on the stack weights agrees with the invite label in 6/6
